In [3]:
using Revise
using SymScan
using LinearAlgebra

In [4]:
pathof(SymScan)

"/Users/fernandopenaranda/Documents/Work/PostdocDonosti/Packages/SymScan.jl/src/SymScan.jl"

#### General Remarks
The lattice vectors and construction of the Hamiltonian implicitly assumes the conventions adopted in Crystalline.jl that in turn follows those of Bilbao Crystallographic. However, since we are evaluating the spectrum, this constraint is not necessary and h can be built with arbitrary numbers.

### Model

In [5]:
## MODEL - DECOUPLED GRAPHENE LAYERS UP TO NN HOPPINGS

function h(k, t = 1,  tc = 0.5)
    a = 1
    a1 = a/2 * [3, √3, 0] # Γ - M line along the x axis 
    a2 = a/2 * [3, -√3, 0]
    a3 = a * [0, 0, 1]
    δ1 = a/2 * [1, √3, 0]
    δ2 = a/2 * [1, -√3, 0]
    δ3 = a * [-1, 0, 0]
    ds = [δ1,δ2,δ3]
    fk = f(k, ds) # first neighbour hoppings
    nnk = real(nn(k, ds)) # second neighbour hoppings 
    return [tc*nnk  t*fk; conj(t*fk)  tc*nnk ]
end

f(k, deltas) = sum(exp(im * dot(k, δ)) for δ in deltas)
nn(k, deltas)= nn(k, deltas[1], deltas[2], deltas[3])
nn(k, d1, d2, d3) = cis(dot(k,d1-d2)) + cis(dot(k,d2-d3)) + cis(dot(k,d3-d1)) +
    cis(dot(k,-d1+d2)) + cis(dot(k,-d2+d3)) + cis(dot(k,-d3+d1))

function lattice_vectors(a = 1)
    a1 = a/2 * [3, √3, 0]
    a2 = a/2 * [3, -√3, 0]
    a3 = a * [0, 0, 1]
    Rs = [a1,a2,a3]
    return Rs
end


lattice_vectors (generic function with 2 methods)

### Compatibility Symmetry Checks

In [22]:
# Trivial group in Hermann–Mauguin notation
Rs = lattice_vectors()
magnetic_point_group = "R3"
preliminary_symmetry_check(h, Rs, magnetic_point_group)

1′:  ✔
3₀₀₁⁺′:  ✔
3₀₀₁⁻′:  ✔
{1|⅔,⅓,⅓}′:  ✔
{3₀₀₁⁺|⅔,⅓,⅓}′:  ✔
{3₀₀₁⁻|⅔,⅓,⅓}′:  ✔
{1|⅓,⅔,⅔}′:  ✔
{3₀₀₁⁺|⅓,⅔,⅔}′:  ✔
{3₀₀₁⁻|⅓,⅔,⅔}′:  ✔


In [21]:
# Trivial group in Schönflies notation
magnetic_point_group = "C3h"
preliminary_symmetry_check(h, Rs, magnetic_point_group)

1′:  ✔
3₀₀₁⁺′:  ✔
3₀₀₁⁻′:  ✔
-6₀₀₁⁺′:  ✔
m₀₀₁′:  ✔
-6₀₀₁⁻′:  ✔


In [20]:
# Non-trivial magnetic group in Schönflies notation
magnetic_point_group = "P4′/mmm′"
preliminary_symmetry_check(h, Rs, magnetic_point_group)

1′:  ✔
4₀₀₁⁺:  ✖
4₀₀₁⁻:  ✖
2₁₀₀′:  ✖
2₀₁₀′:  ✖
2₀₀₁′:  ✔
2₁₁₀:  ✔
2₋₁₁₀:  ✔
-1′:  ✔
-4₀₀₁⁺:  ✖
-4₀₀₁⁻:  ✖
m₁₀₀′:  ✖
m₀₁₀′:  ✖
m₀₀₁′:  ✔
m₁₁₀:  ✔
m₋₁₁₀:  ✔


In [19]:
# Wrong input - Read the recommendations
magnetic_point_group = "P/m2'"
preliminary_symmetry_check(h, Rs, magnetic_point_group)

ArgumentError: ArgumentError: Syntax for the Magnetic Point Group not recognized. 
            Did you mean one of: ["Pmm2", "P2", "P2′", "P_C2", "Pm", "Pm1′", "Pm′", "P2/m", "P2/m1′", "P2/m′"]

In [58]:
magnetic_point_group = "Pm"
preliminary_symmetry_check(h, Rs, magnetic_point_group)

1′:  ✔
m₀₁₀′:  ✔


In [15]:
SymScan.test_magnetic_point_group_syntax("D3h")

(1439, "P-6m2")

In [18]:
SymScan.mspacegroup(1439)

MSpaceGroup{3} ⋕187.209 (P-6m2) with 12 operations:
 1′
 3₀₀₁⁺′
 3₀₀₁⁻′
 2₂₁₀′
 2₁₂₀′
 2₋₁₁₀′
 -6₀₀₁⁺′
 m₀₀₁′
 -6₀₀₁⁻′
 m₁₀₀′
 m₁₁₀′
 m₀₁₀′